# 1. Header, Citations, Purpose & Topic Bridges

**Topic:** Input Convex Neural Networks (ICNN), Brenier Optimal Transport, and Certified Monotonic Networks
**Category:** 06_smooth_and_physics_informed_ai

### Citations & Core References
- Amos, B., Xu, L., & Kolter, J. Z. (2017). *Input Convex Neural Networks*. ICML.
- Brenier, Y. (1991). *Polar Factorization and Monotone Rearrangement of Vector-Valued Functions*. Archive for Rational Mechanics and Analysis.
- Makkuva, A., et al. (2020). *Optimal Transport Mapping via Input Convex Neural Networks*. ICML.

## 2. Mathematical Foundations & Governing Equations

### Input Convex Neural Network Potential
$$\psi(x) = \text{softplus}(W_{z1})^T \text{ReLU}(x W_{x0} + b_0) + x W_{x1} + b_1, \quad \nabla_x^2 \psi(x) \succeq 0$$

### Brenier Optimal Transport Map & Monotonicity
$$T(x) = x + \nabla_x \psi(x)$$
$$\langle T(x_1) - T(x_2), x_1 - x_2 \rangle \ge \|x_1 - x_2\|_2^2$$

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import torch
import torch.nn.functional as F

torch.manual_seed(42)

# Parameters: B=3, D=4, D_h=8
B, D, D_h = 3, 4, 8
x_input = torch.randn(B, D)
w_x0 = torch.randn(D, D_h) * 0.2
b0 = torch.randn(D_h) * 0.1
w_z1 = torch.randn(D_h, 1) * 0.2
w_x1 = torch.randn(D, 1) * 0.2
b1 = torch.randn(1) * 0.1
print(f"Setup complete: x_input {x_input.shape}, w_x0 {w_x0.shape}, w_z1 {w_z1.shape}")

In [ ]:
# 4. First-Principles Implementation & Native Parity
def icnn_potential(x, w_x0, b0, w_z1, w_x1, b1):
    w_z1_pos = F.softplus(w_z1)
    z0 = F.relu(x @ w_x0 + b0)
    return z0 @ w_z1_pos + x @ w_x1 + b1

def brenier_ot_gradient_map(x, w_x0, b0, w_z1, w_x1):
    w_z1_pos = F.softplus(w_z1)
    pre_act = x @ w_x0 + b0
    mask = (pre_act > 0.0).to(x.dtype)
    effective_z = mask * w_z1_pos.t()
    grad_psi = effective_z @ w_x0.t() + w_x1.t()
    return x + grad_psi

psi = icnn_potential(x_input, w_x0, b0, w_z1, w_x1, b1)
t_map = brenier_ot_gradient_map(x_input, w_x0, b0, w_z1, w_x1)
print(f"ICNN potential psi: {psi.shape}, Brenier map T(x): {t_map.shape}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification
# Invariant 1: Autograd exact parity with closed-form gradient
x_req = x_input.clone().detach().requires_grad_(True)
psi_eval = icnn_potential(x_req, w_x0, b0, w_z1, w_x1, b1)
grad_autograd = torch.autograd.grad(psi_eval.sum(), x_req)[0]
t_map_autograd = x_req + grad_autograd
diff_grad = torch.max(torch.abs(t_map - t_map_autograd)).item()
assert diff_grad < 1e-6, f"Autograd gradient parity mismatch: {diff_grad}"

# Invariant 2: Certified monotonicity: <T(x1) - T(x2), x1 - x2> >= ||x1 - x2||^2
x1 = x_input[0:1]
x2 = x_input[1:2]
t1 = brenier_ot_gradient_map(x1, w_x0, b0, w_z1, w_x1)
t2 = brenier_ot_gradient_map(x2, w_x0, b0, w_z1, w_x1)
inner_prod = torch.sum((t1 - t2) * (x1 - x2)).item()
dist_sq = torch.sum((x1 - x2) ** 2).item()
assert inner_prod >= dist_sq - 1e-6, f"Monotonicity invariant violated: {inner_prod} < {dist_sq}"
print("All mathematical invariants (Brenier gradient autograd & monotonicity) verified.")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

iters = 100
t0 = time.perf_counter()
for _ in range(iters):
    _ = icnn_potential(x_input, w_x0, b0, w_z1, w_x1, b1)
    _ = brenier_ot_gradient_map(x_input, w_x0, b0, w_z1, w_x1)
dt = (time.perf_counter() - t0) / iters * 1000.0
print(f"Fused ICNN forward & gradient latency: {dt:.3f} ms / step")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation
# Pathological: unconstrained (negative) hidden weights destroy convexity
w_z1_negative = -torch.abs(w_z1) - 0.5
# Numerical second difference along line segment: f(0.5 x1 + 0.5 x2) <= 0.5 f(x1) + 0.5 f(x2)
x_mid = 0.5 * (x1 + x2)
f_mid = icnn_potential(x_mid, w_x0, b0, w_z1_negative, w_x1, b1).item()
f_avg = 0.5 * (icnn_potential(x1, w_x0, b0, w_z1_negative, w_x1, b1).item() + 
               icnn_potential(x2, w_x0, b0, w_z1_negative, w_x1, b1).item())
is_concave = f_mid > f_avg + 1e-4
print(f"Negative-weight network violates convexity: {is_concave} (f_mid={f_mid:.4f} > f_avg={f_avg:.4f})")

# Corrected: Softplus reparameterization guarantees W_z >= 0 and strict convexity
f_mid_c = icnn_potential(x_mid, w_x0, b0, w_z1, w_x1, b1).item()
f_avg_c = 0.5 * (icnn_potential(x1, w_x0, b0, w_z1, w_x1, b1).item() + 
                 icnn_potential(x2, w_x0, b0, w_z1, w_x1, b1).item())
assert f_mid_c <= f_avg_c + 1e-6, "Corrected ICNN failed convexity midpoint inequality"
print(f"Corrected ICNN satisfies convexity: f_mid={f_mid_c:.4f} <= f_avg={f_avg_c:.4f}")

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Convexity by Design**: Non-negative hidden weights with non-decreasing convex activations guarantee input convexity globally.
2. **Optimal Transport**: Brenier's theorem turns optimal transport into convex potential estimation via $T(x) = \nabla \psi(x)$.
3. **Certified Monotonicity**: The gradient of an ICNN is mathematically guaranteed to be monotonic, satisfying safety and regulatory requirements.